# 08 · Accelerate — M×N GPU (subprocess.Popen, wheel 모듈 모드)

`accelerate launch` CLI 를 `subprocess.Popen` 으로 driver 에서 띄운다. **시스템의 `/databricks/python3/bin/accelerate` 가 wheel 로 설치된 `recommender_pkg` 를 보지 못하는 문제** 때문에, notebook-scoped Python(`sys.executable`)로 `accelerate.commands.accelerate_cli` 를 모듈 실행하고, 학습 entry 도 `-m recommender_pkg.torch_distributor_trainer` 로 모듈 모드를 강제한다. 자세한 배경은 [`torch_distributor_trainer.py`](https://github.com/Aiden-Jeon/databricks-distributed-training/blob/main/03-custom-package-script-based/custom_packages/src/recommender_pkg/torch_distributor_trainer.py) 모듈 docstring 참조.

`--num_processes` 는 지정하지 않는다. Accelerate 가 가시 GPU 수만큼 자동으로 띄운다. 토폴로지 1×1 / 1×N / M×N 은 클러스터 사양으로만 결정되며, 이 노트북 한 개로 전부 커버한다.

**클러스터**: Classic GPU, DBR 17.3 LTS ML. 단일 fat-node 또는 multi-node 환경에서 그대로 동작. **사전조건**: `00-setup` 에서 `recommender_pkg` wheel 이 설치되어 있어야 한다.

In [ ]:
%run ./00-setup

## 패키지 import 확인

In [ ]:
import recommender_pkg
print(f"recommender_pkg v{recommender_pkg.__version__}")

## MLflow run 생성

driver 에서 run 을 시작하고 `run_id` 만 빼서 자식 프로세스에 넘긴다. rank 0 자식이 같은 run 에 attach 해서 system metrics / per-epoch 메트릭을 누적한다.

In [ ]:
import os
import mlflow

cfg = CONFIG
mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="acc-MxN", log_system_metrics=True) as run:
    RUN_ID = run.info.run_id
print(f"RUN_ID={RUN_ID}")

## `accelerate launch` 명령 + subprocess.Popen

1. `sys.executable -m accelerate.commands.accelerate_cli launch -m recommender_pkg.torch_distributor_trainer` 형태로 조립 — wheel env 인식 + 패키지 모듈 모드 강제.
2. `sub_env` 에 `DATABRICKS_HOST` / `DATABRICKS_TOKEN` 주입 — 자식 프로세스는 노트북 auth context 를 상속하지 않는다.
3. Py4J keepalive 스레드 — 장시간 학습 중 driver Py4J gateway 가 끊겨 후처리가 폭사하는 것을 막기 위해 5분마다 `SELECT 1` 을 찌른다.
4. `stderr=STDOUT` + `bufsize=1` + `text=True` 로 stdout/stderr 한 스트림 라인 단위 출력.
5. non-zero return 시 `dbutils.notebook.exit` 로 실패 종료 → DAB job retry 가 동작.

In [ ]:
import shlex
import subprocess
import sys
import threading

tower_hidden_args = " ".join(str(x) for x in cfg["tower_hidden"])
ckpt_path = os.path.join(CKPT_DIR, "acc-MxN.pt")

inference_cmd = f"""{sys.executable} -m accelerate.commands.accelerate_cli launch --mixed_precision=bf16 \
    -m recommender_pkg.torch_distributor_trainer \
    --run_id '{RUN_ID}' \
    --db_host '{DB_HOST}' \
    --db_token '{DB_TOKEN}' \
    --data_dir '{DATA_DIR}' \
    --ckpt_path '{ckpt_path}' \
    --n_users {cfg['n_users']} \
    --n_items {cfg['n_items']} \
    --emb_dim {cfg['emb_dim']} \
    --tower_hidden {tower_hidden_args} \
    --batch_size {cfg['batch_size_per_gpu']} \
    --num_epochs {cfg['num_epochs']} \
    --max_steps_per_epoch {cfg['max_steps_per_epoch']} \
    --patience {PATIENCE} \
    --min_delta {MIN_DELTA} \
    --topology MxN
"""

sub_env = os.environ.copy()
sub_env["DATABRICKS_HOST"] = DB_HOST
sub_env["DATABRICKS_TOKEN"] = DB_TOKEN

done_event = threading.Event()

def _keepalive(interval=300):
    while not done_event.is_set():
        try:
            spark.sql("SELECT 1").collect()
        except Exception:
            pass
        done_event.wait(interval)

keepalive_thread = threading.Thread(target=_keepalive, daemon=True)
keepalive_thread.start()

p = subprocess.Popen(
    shlex.split(inference_cmd),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=sub_env,
)

for line in p.stdout:
    print(line, end="")

p.wait()
done_event.set()

if p.returncode != 0:
    dbutils.notebook.exit(f"accelerate launch failed: rc={p.returncode}")